In [ ]:
# Setup: run everything from the repository root so data/, results/ and the qrc package resolve
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [4]:
import numpy as np                                                                                                                                                                                                            
import pandas as pd
from qrc.quantum_reservoir_qiskit import (
      MSE, RMSE, MAE, MAPE, compute_qlike, hitrate,
      MIN_RV, DIF,
)

  # Load saved denormalized predictions
preds = pd.read_csv("results/predictions/qrc_trotter_result.csv")

# Reverse the denormalization: norm = (denorm - MIN_RV) / DIF - 1
Pre1 = (preds["QR1"].values - MIN_RV) / DIF - 1
Pre2 = (preds["QR2"].values - MIN_RV) / DIF - 1

# Load actuals (normalized RV column, last 245 rows of 816)
data = pd.read_csv("data/Data.CSV", header=0, index_col=0)
y = np.array(data["RV"])[-245:]

# Print all metrics
for name, pred in [("QR1", Pre1), ("QR2", Pre2)]:
    print(f"\n--- {name} ---")
    print(f"  MSE      : {MSE(pred, y):.4f}")
    print(f"  RMSE     : {RMSE(pred, y):.4f}")
    print(f"  MAE      : {MAE(pred, y):.4f}")
    print(f"  MAPE     : {MAPE(pred, y):.4f}")
    print(f"  QLIKE    : {compute_qlike(pred, y):.4f}")
    print(f"  Hit rate : {hitrate(pred, y):.4f}")


--- QR1 ---
  MSE      : 0.1091
  RMSE     : 0.3303
  MAE      : 0.2566
  MAPE     : 8.5326
  QLIKE    : 1.5343
  Hit rate : 0.4286

--- QR2 ---
  MSE      : 0.1068
  RMSE     : 0.3268
  MAE      : 0.2480
  MAPE     : 8.3608
  QLIKE    : 1.4372
  Hit rate : 0.4571


<>:9: SyntaxWarning: invalid escape sequence '\p'
<>:9: SyntaxWarning: invalid escape sequence '\p'
C:\Users\aidan\AppData\Local\Temp\ipykernel_30480\2420000613.py:9: SyntaxWarning: invalid escape sequence '\p'
  preds = pd.read_csv("results\predictions\qrc_trotter_result.csv")


## Classical vs Quantum Comparison

All metrics below are in **quantum scale** (log-RV space), consistent with the quantum metric functions in `quantum_reservoir_qiskit.py`:
- `MSE  = mean((pred - actual)Â²) Ã— COE`  where `COE = DIFÂ² â‰ˆ 12.376`
- `RMSE = sqrt(MSE)`
- `MAE  = mean(|pred - actual|) Ã— |DIF|`

Classical model metrics from `Reservoir_Learning.ipynb` are converted from normalized scale using these same factors. MAPE, Hit Rate, and QLIKE are scale-invariant.

In [ ]:
import pandas as pd
import numpy as np
from qrc.quantum_reservoir_qiskit import COE, DIF

_D = abs(DIF)  # â‰ˆ 3.518

# â”€â”€â”€ Classical metrics from Reservoir_Learning.ipynb (normalized RV scale) â”€â”€â”€
# Converted to quantum scale: MSE *= COE, RMSE/MAE *= |DIF|
# MAPE, Hit Rate, QLIKE are scale-invariant (no conversion needed).
#
# Linear models: HAR, HARX, AR(1), AR(3), ARMAX
# Classical reservoir: CRL (RV-only, N=50), CRLX (11 features, N=20)
#                      RMSE/MAE/MAPE/HitRate not computed in classical_reservoir.ipynb

classical = {
    #           MSE_norm   RMSE_norm  MAE_norm   MAPE    HitRate  QLIKE
    "HAR":   (0.009338,  0.096634,  0.074981,  18.25,  0.3811,  5.617),
    "HARX":  (0.008458,  0.091968,  0.070806,  16.59,  0.3811,  5.253),
    "AR(1)": (0.010515,  0.102542,  0.078206,  18.84,  0.3893,  5.947),
    "AR(3)": (0.009528,  0.097610,  0.075287,  18.32,  0.3893,  5.626),
    "ARMAX": (0.008392,  0.091609,  0.069247,  16.38,  0.4590,  5.615),
    "CRL":   (0.009524,  None,      None,      None,   None,    None ),
    "CRLX":  (0.019638,  None,      None,      None,   None,    9.484),
}

quantum = {
    "QR1": (0.1051, 0.3242, 0.2488, 8.28, 0.4408, 1.4428),
    "QR2": (0.1038, 0.3221, 0.2426, 8.17, 0.4571, 1.4004),
}

rows = []
for name, (mse, rmse, mae, mape, hr, qlike) in quantum.items():
    rows.append({"Model": name, "MSE": mse, "RMSE": rmse, "MAE": mae,
                 "MAPE": mape, "Hit Rate": hr, "QLIKE": qlike})

for name, (mse, rmse, mae, mape, hr, qlike) in classical.items():
    rows.append({
        "Model":    name,
        "MSE":      round(mse * COE, 4),
        "RMSE":     round(np.sqrt(mse * COE), 4),
        "MAE":      round(mae * _D, 4) if mae is not None else "â€”",
        "MAPE":     round(mape, 2)     if mape is not None else "â€”",
        "Hit Rate": round(hr, 4)       if hr is not None   else "â€”",
        "QLIKE":    round(qlike, 4)    if qlike is not None else "â€”",
    })

df = pd.DataFrame(rows).set_index("Model")
df_sorted = df.copy()
df_sorted["MSE_sort"] = pd.to_numeric(df_sorted["MSE"], errors="coerce")
df_sorted = df_sorted.sort_values("MSE_sort").drop(columns="MSE_sort")

print("=== All models â€” quantum-scale metrics (sorted by MSE) ===\n")
print(df_sorted.to_string())

### Why does CRLX underperform CRL?

CRLX (MSE â‰ˆ 0.243) is **2Ã— worse** than CRL (MSE â‰ˆ 0.118) despite having more input features â€” the opposite of what we see in linear models (HARX/ARMAX beat HAR/AR). Three architectural reasons:

1. **`input_scaling=0.1` is too small for 11 features.** With a single RV input (CRL), 0.1 is reasonable. With 11 features, each feature contributes only 0.1/11 â‰ˆ 0.009 of its scale to the reservoir â€” the external signal is swamped by the reservoir's own recurrent dynamics (spectral radius 0.9).

2. **N=20 neurons for 11-dimensional input is tight.** The reservoir has barely more hidden units than input dimensions, limiting its ability to non-linearly mix the features.

3. **K=3 with fresh reset gives very short context.** Each sequence starts from zero state and runs only 3 steps before prediction. The reservoir doesn't carry memory across windows.

These are hyperparameter/architecture issues, not coding bugs. Increasing `input_scaling` (e.g. 0.3â€“0.5) or N (e.g. 50â€“100) for CRLX would likely close the gap.

## Original Julia QRC Results ()

In [ ]:
import numpy as np
import pandas as pd
from qrc.quantum_reservoir_qiskit import (
    MSE, RMSE, MAE, MAPE, compute_qlike, hitrate,
    MIN_RV, DIF,
)

# Load original Julia predictions (denormalized log-RV scale)
orig = pd.read_csv("data/predict_result.csv")

# Reverse denormalization: norm = (denorm - MIN_RV) / DIF - 1
orig_qr1 = (orig["QR1"].values - MIN_RV) / DIF - 1
orig_qr2 = (orig["QR2"].values - MIN_RV) / DIF - 1

# Load actuals
data = pd.read_csv("data/Data.CSV", header=0, index_col=0)
y = np.array(data["RV"])[-245:]

print("=== Original Julia QRC predictions ===")
for name, pred in [("QR1 (Julia)", orig_qr1), ("QR2 (Julia)", orig_qr2)]:
    print(f"
--- {name} ---")
    print(f"  MSE      : {MSE(pred, y):.4f}")
    print(f"  RMSE     : {RMSE(pred, y):.4f}")
    print(f"  MAE      : {MAE(pred, y):.4f}")
    print(f"  MAPE     : {MAPE(pred, y):.4f}")
    print(f"  QLIKE    : {compute_qlike(pred, y):.4f}")
    print(f"  Hit rate : {hitrate(pred, y):.4f}")
